# Test Suite: Text Preprocessing & Stylometric Linguistics
This notebook executes the unit tests and interactive validations for `src/preprocessing.py`.
It tests:
- HTML and URL stripping
- Contraction expansion (`don't` -> `do not`, `won't` -> `will not`)
- Repeated character normalization (`sooooo` -> `soo`)
- **Negation preservation in stopwords** (`not`, `never`, `no` are protected)
- Stemming / lemmatization
- Extraction of 9 stylometric markers (word count, punctuation ratio, uppercase ratio, etc.)
- Scikit-learn transformer compatibility (`TextPreprocessorTransformer`)


In [1]:
import os
import sys
import unittest
import pandas as pd

# Ensure project root is in sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

print(f"Working Directory: {os.getcwd()}")
from src.preprocessing import (
    clean_html,
    clean_urls,
    expand_contractions,
    normalize_repeated_chars,
    clean_text_light,
    clean_text_standard,
    clean_text_lemmatized,
    extract_linguistic_features,
    TextPreprocessorTransformer
)
print("Preprocessing module imported successfully!")


Working Directory: e:\ML\mini project 2
Preprocessing module imported successfully!


## Interactive Demonstration 1: HTML, URL & Contraction Cleaning


In [2]:
raw_text = "Check out <a href='https://amazon.com'>this link</a>! I don't think it won't work, soooo goooood!!"
print(f"Original Text: {raw_text}")
no_html = clean_html(raw_text)
print(f"After clean_html: {no_html}")
no_url = clean_urls(no_html)
print(f"After clean_urls: {no_url}")
expanded = expand_contractions(no_url)
print(f"After expand_contractions: {expanded}")
normalized = normalize_repeated_chars(expanded)
print(f"After normalize_repeated_chars: {normalized}")


Original Text: Check out <a href='https://amazon.com'>this link</a>! I don't think it won't work, soooo goooood!!
After clean_html: Check out  this link ! I don't think it won't work, soooo goooood!!
After clean_urls: Check out  this link ! I don't think it won't work, soooo goooood!!
After expand_contractions: Check out  this link ! I do not think it will not work, soooo goooood!!
After normalize_repeated_chars: Check out  this link ! I do not think it will not work, soo good!!


## Interactive Demonstration 2: Negation Preservation in Stopword Filtering
Crucial test: `not` and `never` must be preserved to prevent sentiment polarity inversion.


In [3]:
test_sentence = "This was not a good purchase. I would never buy again."
cleaned_preserved = clean_text_standard(test_sentence, preserve_negations=True)
cleaned_standard = clean_text_standard(test_sentence, preserve_negations=False)

print(f"Original Text:           '{test_sentence}'")
print(f"Preserving Negations:    '{cleaned_preserved}'  <- 'not' and 'never' KEPT!")
print(f"Standard Stopwords:      '{cleaned_standard}'   <- 'not' and 'never' stripped")


Original Text:           'This was not a good purchase. I would never buy again.'
Preserving Negations:    'not good purchase would never buy'  <- 'not' and 'never' KEPT!
Standard Stopwords:      'good purchase would never buy'   <- 'not' and 'never' stripped


## Interactive Demonstration 3: Stylometric Feature Extraction
Extract 9 domain-engineered linguistic and stylistic metrics from text.


In [4]:
sample_review = "WOW!! Is this REAL? I bought it for my family."
feats = extract_linguistic_features(sample_review)
for k, v in feats.items():
    print(f"  {k:25s}: {v}")


  char_count               : 46.0
  word_count               : 10.0
  avg_word_length          : 3.3
  exclamation_count        : 2.0
  question_count           : 1.0
  uppercase_ratio          : 0.1957
  punctuation_ratio        : 0.087
  first_person_pronouns    : 2.0
  lexical_diversity        : 1.0


## Formal Unittest Suite Runner


In [5]:
class TestPreprocessing(unittest.TestCase):

    def test_clean_html(self):
        html_text = "This is a <b>great</b> product!<br/><p>Highly recommended.</p>"
        cleaned = clean_html(html_text)
        self.assertNotIn("<b>", cleaned)
        self.assertNotIn("<br/>", cleaned)
        self.assertNotIn("</p>", cleaned)
        self.assertIn("great", cleaned)

    def test_clean_urls(self):
        url_text = "Check out https://amazon.com/dp/12345 or www.review.org for details."
        cleaned = clean_urls(url_text)
        self.assertNotIn("https://", cleaned)
        self.assertNotIn("www.", cleaned)
        self.assertIn("details", cleaned)

    def test_expand_contractions(self):
        text = "I don't think it won't work, but it wasn't broken."
        expanded = expand_contractions(text)
        self.assertIn("do not", expanded)
        self.assertIn("will not", expanded)
        self.assertIn("was not", expanded)

    def test_normalize_repeated_chars(self):
        text = "This is soooo coooool and veeeery goooood"
        normalized = normalize_repeated_chars(text)
        self.assertNotIn("soooo", normalized)
        self.assertIn("soo", normalized)
        self.assertIn("good", normalized)

    def test_negation_preservation(self):
        text = "This was not a good purchase. I would never buy again."
        
        # When preserving negations:
        cleaned_with_neg = clean_text_standard(text, preserve_negations=True)
        self.assertIn("not", cleaned_with_neg.split())
        self.assertNotIn("was", cleaned_with_neg.split())
        self.assertNotIn("this", cleaned_with_neg.split())

        # When NOT preserving negations:
        cleaned_no_neg = clean_text_standard(text, preserve_negations=False)
        self.assertNotIn("not", cleaned_no_neg.split())

    def test_lemmatization(self):
        text = "He loved running and buying products"
        lemmatized = clean_text_lemmatized(text)
        self.assertIn("love", lemmatized)
        self.assertIn("run", lemmatized)
        self.assertIn("buy", lemmatized)
        self.assertIn("product", lemmatized)

    def test_extract_linguistic_features(self):
        text = "WOW!! Is this REAL? I bought it for my family."
        feats = extract_linguistic_features(text)
        self.assertEqual(feats["exclamation_count"], 2.0)
        self.assertEqual(feats["question_count"], 1.0)
        self.assertGreater(feats["uppercase_ratio"], 0.1)
        self.assertGreater(feats["first_person_pronouns"], 1.0)
        self.assertGreater(feats["lexical_diversity"], 0.5)

    def test_sklearn_transformer(self):
        transformer = TextPreprocessorTransformer(strategy="standard", preserve_negations=True)
        series = pd.Series(["I didn't like this product!", "Great quality!"])
        transformed = transformer.fit_transform(series)
        self.assertEqual(len(transformed), 2)
        self.assertIn("not", transformed.iloc[0])

# Run unittest suite in notebook
suite = unittest.TestLoader().loadTestsFromTestCase(TestPreprocessing)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
assert result.wasSuccessful(), "Some tests failed!"


test_clean_html (__main__.TestPreprocessing.test_clean_html) ... ok
test_clean_urls (__main__.TestPreprocessing.test_clean_urls) ... ok
test_expand_contractions (__main__.TestPreprocessing.test_expand_contractions) ... ok
test_extract_linguistic_features (__main__.TestPreprocessing.test_extract_linguistic_features) ... ok
test_lemmatization (__main__.TestPreprocessing.test_lemmatization) ... ok
test_negation_preservation (__main__.TestPreprocessing.test_negation_preservation) ... ok
test_normalize_repeated_chars (__main__.TestPreprocessing.test_normalize_repeated_chars) ... ok
test_sklearn_transformer (__main__.TestPreprocessing.test_sklearn_transformer) ... ok

----------------------------------------------------------------------
Ran 8 tests in 2.105s

OK
